# Machine Learning Report
## Hotel Occupancy Prediction for Antalya/Alanya Resort Hotels Using Google Trends

**Prepared by:** Bedirhan Sar  \n**Project:** Hotel Occupancy Analysis and Prediction

---

## Purpose of the ML Stage

The EDA phase showed that:
- occupancy is strongly seasonal,
- same-day Google Trends relationships are generally weak,
- several lagged Google Trends features are more promising,
- and the strongest signals come mainly from selected Türkiye and Germany keywords.

The purpose of the ML stage is therefore not to replace the EDA logic, but to test a focused predictive question:

> Do lagged Google Trends features improve hotel occupancy prediction beyond hotel identity, calendar seasonality, and recent occupancy history?

As the modeling stage developed, a second and more practical question also became necessary:

> Do the learned models outperform simple time-series benchmark rules such as persistence?


## Modeling Strategy Used So Far

Two learned modeling settings were compared.

### 1. Baseline model
Feature set:
- hotel identity,
- calendar features,
- cyclical seasonality encoding,
- occupancy lags at 7, 14, and 28 days.

This setup captures the main internal structure of the problem: hotel-specific level differences, seasonality, and autoregressive behavior.

### 2. Baseline + lagged Google Trends model
This setup includes everything in the baseline model, plus selected lagged Google Trends variables chosen from the EDA stage.

The intention is to test whether Trends adds **incremental signal** beyond what occupancy history and seasonality already explain.

### 3. Rule-based time-series benchmarks
To judge whether the learned models are truly useful, two simple benchmark rules were also added later:
- **NaivePersistence**: predict today's occupancy using yesterday's occupancy for the same hotel
- **SeasonalNaive7**: predict today using occupancy from 7 days earlier

These are not competitors to the EDA logic. They are evaluation references that help interpret whether the ML pipeline is actually beating a simple baseline.


## Feature Engineering

### Calendar and seasonality features
The following features were used to capture recurring temporal structure:
- `month`
- `day_of_week`
- `week_of_year`
- `doy_sin`
- `doy_cos`

### Autoregressive occupancy features
Recent hotel behavior was represented with:
- `occupancy_lag_7`
- `occupancy_lag_14`
- `occupancy_lag_28`

The naive benchmark script also used:
- `occupancy_lag_1` for persistence
- `occupancy_lag_7` for seasonal naive prediction

### Lagged Google Trends features
The strongest feature-lag combinations from the earlier EDA pass were used as a compact first-pass Trends feature set.

Important correction applied in the latest version:

Because Google Trends variables are **date-level shared signals**, their lagged versions must be created on a **unique date table first** and then merged back by date. They should not be shifted directly on the hotel-date modeling table.

This alignment issue has now been corrected in:
- `scripts/modeling_baseline_commented.py`
- `scripts/hotel_normalization_robustness_commented.py`


## Models Compared

Two learned models were used in the ML comparison:

### Ridge Regression
A regularized linear benchmark.

### Random Forest Regressor
A stronger non-linear benchmark that can model interactions and non-linear relationships without requiring explicit feature transformations.

These were evaluated both against each other and against simple rule-based benchmarks.


## Validation Design

The validation strategy was expanded step by step.

### 1. First-pass temporal holdout
The first ML pass used a **time-aware split** instead of random train/test splitting:
- earlier dates were used for training,
- later dates were used for testing.

This was a reasonable starting point, but it still had one methodological weakness: the baseline and baseline + Trends models did not necessarily use the exact same rows before the split.

### 2. Fair same-window comparison
To fix that issue, a second script was added:

- `scripts/modeling_fair_comparison_commented.py`

This version restricts both model settings to the **exact same rows** and then applies **one shared future test window**. This makes the baseline vs baseline + Trends comparison more defensible.

### 3. Walk-forward validation
To move beyond a single holdout period, a third script was added:

- `scripts/modeling_walk_forward_commented.py`

This version uses **expanding-window walk-forward validation**. The model is trained on earlier dates and then tested on the next future block, repeated across several folds. This checks whether the Trends signal is useful **consistently across time**, not just in one final split.

### 4. Naive benchmark comparison
Finally, a fourth script was added:

- `scripts/modeling_naive_benchmarks_commented.py`

This version evaluates the learned models against simple benchmark rules. This is important because a time-series model should not only beat another learned model; it should also be judged against a realistic naive baseline.


## Current First-Pass ML Findings

After correcting lagged Google Trends alignment and rerunning the first-pass pipeline, the main learned-model result remained consistent:

- the best baseline-only model was weaker,
- the best baseline + lagged Trends model performed better,
- and the best overall learned configuration remained **Random Forest with lagged Google Trends**.

### Updated pooled learned-model result
- Best baseline-only RMSE: **approximately 5.87**
- Best baseline + Trends RMSE: **approximately 4.80**

This shows that lagged Google Trends adds useful predictive information **within the learned-model comparison**.

However, this result should not be treated as the final ML conclusion on its own, because later benchmark analysis shows that simple persistence remains even stronger.


## Hotel-Level Prediction Plots

These figures show actual vs predicted occupancy for the best non-robustness trends-augmented learned model.

### Azura Deluxe
![Azura Deluxe prediction](Figures/actual_vs_pred_azura_deluxe.png)

### Side Mare Hotel
![Side Mare Hotel prediction](Figures/actual_vs_pred_side_mare_hotel.png)


## Hotel-wise Normalization Robustness in the ML Stage

The EDA stage already tested whether pooled correlation findings survive hotel-wise normalization. The ML stage extends that idea by asking:

> If the target is normalized within each hotel, does the incremental value of lagged Google Trends remain visible?

This matters because the two hotels do not operate at exactly the same occupancy level or variability scale.

### Normalized target used in robustness modeling

`target_hotel_z = (occupancy_rate - train_hotel_mean) / train_hotel_std`

Important leakage rule:
- hotel mean and standard deviation were computed from the **train period only**,
- then applied to both train and test.

This keeps the normalization leakage-safe.

### Robustness result
After rerunning the corrected robustness pipeline:
- the best hotel-normalized baseline-only model remained weaker,
- the best hotel-normalized baseline + Trends model still performed better,
- and the best learned model again remained **Random Forest with lagged Google Trends**.

### Updated robustness result (raw scale after back-transformation)
- Best baseline-only RMSE: **approximately 6.19**
- Best baseline + Trends RMSE: **approximately 5.67**

This suggests that the value of lagged Google Trends is **not only an artifact of pooling two hotels with different average occupancy levels**.

Still, this is a robustness check within the learned-model framework. It does not by itself show that the learned models outperform simple persistence-style rules.


## Hotel-normalized Prediction Plots

These figures show actual vs predicted occupancy for the best hotel-normalized robustness model, back-transformed to raw occupancy scale.

### Azura Deluxe
![Azura Deluxe normalized-target robustness prediction](Figures/actual_vs_pred_hotel_z_azura_deluxe.png)

### Side Mare Hotel
![Side Mare Hotel normalized-target robustness prediction](Figures/actual_vs_pred_hotel_z_side_mare_hotel.png)


## Fair Same-Window Validation

The first-pass comparison was useful, but it still allowed the baseline and baseline + Trends models to be built on slightly different non-missing datasets. The fair same-window pass fixes this by forcing both setups to use the **same rows** and the **same test period**.

### Main learned-model result
On this cleaner comparison window:
- best baseline-only RMSE: **4.974**
- best baseline + Trends RMSE: **4.798**

The best learned model again remained **Random Forest**, and the Trends-augmented version still performed better than the learned baseline-only version.

This matters because it shows that the earlier Trends gain does **not disappear** when the comparison is made on a stricter, methodologically cleaner evaluation window.

### Fair same-window prediction plots

#### Azura Deluxe
![Azura Deluxe fair same-window prediction](Figures/actual_vs_pred_same_window_azura_deluxe.png)

#### Side Mare Hotel
![Side Mare Hotel fair same-window prediction](Figures/actual_vs_pred_same_window_side_mare_hotel.png)


## Walk-Forward Validation

The walk-forward stage is a stronger time-series validation step than a single holdout. Instead of testing on only one final block, it evaluates the same learned-model logic across several future periods.

### Why this matters
A single split can sometimes look unusually good or unusually bad depending on the chosen period. Walk-forward validation checks whether the conclusion is **stable across time**.

### Main learned-model result
Across folds, the best average performance again came from the **Random Forest with lagged Google Trends**:

- best baseline-only mean RMSE across folds: **8.166**
- best baseline + Trends mean RMSE across folds: **8.035**

So the average improvement from Trends remained **positive but modest**.

### Interpretation
The walk-forward learned-model results are clearly more variable than the single holdout results. Some folds are much harder than others, and overall performance is less stable across time. This suggests that:
- seasonality and timing effects remain dominant,
- the Google Trends contribution is useful but not large,
- and final claims should rely more on repeated validation than on a single test split.

### Walk-forward validation figures

#### RMSE by fold
![Walk-forward RMSE by fold](Figures/walk_forward_rmse_by_fold.png)

#### Mean RMSE summary
![Walk-forward mean RMSE summary](Figures/walk_forward_mean_rmse_summary.png)


## Naive Benchmark Comparison

At this point, the key remaining question was not whether lagged Google Trends helped the learned models relative to each other. That had already been shown. The more important question became:

> Do the learned models beat simple time-series benchmark rules?

To answer this, two rule-based benchmarks were added:
- **NaivePersistence**: predict today with yesterday's occupancy for the same hotel
- **SeasonalNaive7**: predict today with occupancy from 7 days earlier

### Fair same-window benchmark result
On the same fair comparison window:
- **NaivePersistence RMSE: 4.068**
- **SeasonalNaive7 RMSE: 7.808**
- best learned model (`baseline_plus_trends / RandomForest`) RMSE: **4.798**

This means that while the Trends-augmented Random Forest improved on the learned baseline, it still **did not beat NaivePersistence**.

### Walk-forward benchmark result
Across folds:
- **NaivePersistence mean RMSE: 4.170**
- **SeasonalNaive7 mean RMSE: 8.350**
- best learned model (`baseline_plus_trends / RandomForest`) mean RMSE: **8.035**

Again, NaivePersistence remained the strongest benchmark overall. The learned models, including the Trends-augmented ones, did not surpass it.

### Interpretation
This is one of the most important ML findings in the project. It changes the final interpretation of the modeling stage:
- lagged Google Trends does provide useful signal **inside the learned-model comparison**,
- but the overall forecasting problem is still dominated by short-term persistence,
- and the current learned models do **not outperform** the strongest naive benchmark.

So the honest conclusion is not "Google Trends creates the best forecasting model." The more accurate conclusion is:

> Lagged Google Trends improves the learned models, but the current ML pipeline still fails to beat a simple persistence rule.

### Naive benchmark figures

#### Fair same-window RMSE with naive benchmarks
![Fair same-window with naive benchmarks](Naive_Benchmark/same_window_rmse_with_naive_benchmarks.png)

#### Walk-forward RMSE with naive benchmarks
![Walk-forward RMSE with naive benchmarks](Naive_Benchmark/walk_forward_rmse_with_naive_benchmarks.png)

#### Walk-forward mean RMSE with naive benchmarks
![Walk-forward mean RMSE with naive benchmarks](Naive_Benchmark/walk_forward_mean_rmse_with_naive_benchmarks.png)


## What Has Been Done Correctly

At the current project stage, the ML pipeline already includes several strong design choices:

- time-aware train/test splitting,
- correction of the lagged Google Trends alignment bug,
- separate comparison of baseline vs baseline + Trends,
- hotel-level lag creation for occupancy,
- pooled and hotel-level evaluation,
- robustness analysis with hotel-wise normalized target,
- train-only normalization for the robustness target,
- fair same-window comparison on the same rows and same test period,
- walk-forward validation across multiple future folds,
- explicit comparison against naive and seasonal naive benchmarks,
- back-transformed RMSE reporting for business interpretability.


## Current Limitations of the ML Stage

Although the ML stage is now much stronger than the initial first pass, it is still not the final modeling design.

### 1. The learned models do not beat NaivePersistence
This is the most important limitation. The current learned pipeline improves over the learned baseline, but it still cannot beat the strongest simple benchmark.

### 2. Walk-forward performance is still unstable
The walk-forward results vary substantially across folds. This means predictive difficulty changes across time periods, so final claims should stay cautious.

### 3. Feature refinement is still incomplete
The current Trends feature set is still a compact first-pass selection. A more systematic feature refinement stage is still needed.

### 4. External signal set is still narrow
Google Trends is useful, but it is still only one external signal family. One carefully chosen external dataset may still improve the project.

### 5. The target may simply be very persistence-dominated
Daily occupancy may be driven so strongly by short-run continuity that additional signals mainly provide secondary refinements rather than major forecasting gains.


## Interpretation

The ML evidence now supports a more careful and more honest conclusion than before:

- **seasonality and recent occupancy remain the dominant drivers**,
- **lagged Google Trends adds incremental predictive value** inside the learned-model framework,
- this signal survives the first-pass, robustness, fair same-window, and walk-forward learned-model comparisons,
- but the current learned models still **do not outperform NaivePersistence**,
- and under repeated validation the Trends gain remains **positive but modest** and less stable across time.

So the ML stage is aligned with the EDA logic, but the final modeling claim must stay limited:
- Google Trends is not a strong same-day demand proxy,
- selected lagged search signals can act as an **early supporting indicator**,
- they improve the learned models somewhat,
- but they do not yet produce a forecasting system stronger than simple persistence.


## Recommended Next ML Steps

The next clean steps for the project are:

1. refine feature selection using only properly aligned lagged features,  
2. inspect feature importance and stability across folds,  
3. investigate why persistence is so strong and whether a different forecasting horizon would be more informative,  
4. decide whether one additional external dataset is worth adding,  
5. summarize final conclusions in a presentation-ready results table that clearly distinguishes learned-model gains from benchmark-relative performance.


## Related Files

Main scripts:
- `scripts/modeling_baseline_commented.py`
- `scripts/hotel_normalization_robustness_commented.py`
- `scripts/modeling_fair_comparison_commented.py`
- `scripts/modeling_walk_forward_commented.py`
- `scripts/modeling_naive_benchmarks_commented.py`

Main figure folders used in this report:
- `ML/Figures/`
- `ML/Naive_Benchmark/`

Main output folders:
- `model_outputs/baseline_ml/`
- `model_outputs/hotel_normalization_robustness/`
- `model_outputs/fair_same_window_comparison/`
- `model_outputs/walk_forward_validation/`
- `model_outputs/naive_benchmark_comparison/`

Main report tables:
- `reports/ML_reports/model_comparison.csv`
- `reports/ML_reports/model_comparison_by_hotel.csv`
- `reports/ML_reports/model_comparison_same_window.csv`
- `reports/ML_reports/model_comparison_by_hotel_same_window.csv`
- `reports/ML_reports/walk_forward_fold_results.csv`
- `reports/ML_reports/walk_forward_summary.csv`
- `reports/ML_reports/walk_forward_by_hotel.csv`
- `reports/ML_reports/same_window_with_naive_benchmarks.csv`
- `reports/ML_reports/same_window_with_naive_benchmarks_by_hotel.csv`
- `reports/ML_reports/walk_forward_with_naive_benchmarks_fold_results.csv`
- `reports/ML_reports/walk_forward_with_naive_benchmarks_summary.csv`
- `reports/ML_reports/walk_forward_with_naive_benchmarks_by_hotel.csv`


## Current Status

**Status:** First-pass ML completed, lagged Google Trends alignment bug corrected, hotel-normalized robustness completed, fair same-window comparison added, walk-forward validation added, naive benchmark comparison added, final interpretation revised accordingly, and ML report figure paths updated to match the figure folders.
